# Modelado de Fatiga con GRU (Gated Recurrent Unit) Personalizada

Este notebook contiene la explicación teórica, la revisión de literatura científica, la arquitectura detallada, y la implementación paso a paso de una red **GRU implementada manualmente en PyTorch** (sin recurrir al módulo `nn.GRU` estándar) para predecir los niveles continuos de fatiga física y mental del dataset **FatigueSet**.

---

## 1. Fundamentos Teóricos y Literatura de Referencia

La unidad recurrente de compuertas o **Gated Recurrent Unit (GRU)** fue propuesta por Kyunghyun Cho et al. en 2014 como una variación optimizada de la arquitectura LSTM clásica. Su principal objetivo es simplificar la estructura recurrente al eliminar la celda de memoria interna independiente ($c_t$), unificando el estado oculto y reduciendo la cantidad total de compuertas matemáticas de cuatro a dos (puertas de actualización y de reinicio).

Esta simplificación reduce significativamente el número de parámetros libres a optimizar, resultando en un entrenamiento computacionalmente más rápido y eficiente, especialmente en datasets con secuencias temporales de tamaño moderado o recursos computacionales limitados.

### Formulación Matemática

Para un paso de tiempo $t$, dada la entrada actual $x_t$ y el estado oculto anterior $h_{t-1}$, la celda calcula:

1. **Puerta de Actualización (Update Gate):** Determina cuánta información del estado oculto anterior debe propagarse al futuro.
   $$z_t = \sigma(W_z x_t + U_z h_{t-1} + b_z)$$

2. **Puerta de Reinicio (Reset Gate):** Determina cuánta de la información pasada se debe ignorar o descartar.
   $$r_t = \sigma(W_r x_t + U_r h_{t-1} + b_r)$$

3. **Estado Oculto Candidato (Candidate Hidden State):** Representa el nuevo contenido de información a almacenar, combinando la entrada actual y la memoria pasada atenuada por la puerta de reinicio.
   $$\tilde{h}_t = \tanh(W_h x_t + U_h (r_t \odot h_{t-1}) + b_h)$$

4. **Estado Oculto Final (Hidden State Update):** Modula linealmente la mezcla entre el estado anterior y el candidato mediante la compuerta de actualización.
   $$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

Donde:
* $\sigma(z) = \frac{1}{1 + e^{-z}}$ es la función sigmoide.
* $\tanh(z)$ es la tangente hiperbólica.
* $\odot$ representa el producto de Hadamard (multiplicación elemento a elemento).
* $W_*$ y $U_*$ representan las matrices de pesos proyectadas sobre las dimensiones correspondientes, y $b_*$ representa los sesgos.

---

### Diagrama de Flujo de la Celda (Mermaid)

```mermaid
graph TD
    subgraph "Celda Custom GRU (Paso t)"
        xt["Entrada actual: x_t"]
        h_prev["Estado oculto anterior: h_{t-1}"]

        z_t["Update Gate: z_t = σ(W_z x_t + U_z h_{t-1} + b_z)"]
        r_t["Reset Gate: r_t = σ(W_r x_t + U_r h_{t-1} + b_r)"]
        h_tilde["Candidato Oculto: h̃_t = tanh(W_h x_t + U_h (r_t ⊙ h_{t-1}) + b_h)"]

        h_update["Actualización Oculta: h_t = (1 - z_t) ⊙ h_{t-1} + z_t ⊙ h̃_t"]

        xt --> z_t
        xt --> r_t
        xt --> h_tilde

        h_prev --> z_t
        h_prev --> r_t
        h_prev --> h_tilde_mul["r_t ⊙ h_{t-1}"]
        r_t --> h_tilde_mul
        h_tilde_mul --> h_tilde

        h_prev --> h_update
        z_t --> h_update
        h_tilde --> h_update
        
        h_update --> h_out["Salida Oculta: h_t"]
    end
```

---

### Citas Bibliográficas Científicas

* **Cho, K., Van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H., & Bengio, Y. (2014).** *Learning Phrase Representations using RNN Encoder-Decoder for Statistical Machine Translation*. arXiv preprint arXiv:1406.1078. [Enlace al Paper](https://arxiv.org/abs/1406.1078)
* **Chung, J., Gulcehre, C., Cho, K., & Bengio, Y. (2014).** *Empirical evaluation of gated recurrent neural networks on sequence modeling*. arXiv preprint arXiv:1412.3555. [Enlace al Paper](https://arxiv.org/abs/1412.3555) *(Compara experimentalmente GRU con LSTM en varias tareas de modelización de series temporales)*.

In [1]:
# SETUP e IMPORTACIONES
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Añadir fatigueset-lib al sys.path
lib_path = str(Path.cwd().parent / "fatigueset-lib")
if lib_path not in sys.path:
    sys.path.insert(0, lib_path)

from fatigueset import FatigueSetPipeline
from fatigueset.models import CustomGRURegressor, FatigueSequenceDataset
from fatigueset.models.rnn import _prepare_target_table, _merge_raw_streams, _build_sequences

print("[OK] Imports completados y path configurado.")
print(f"Dispositivo actual: {'cuda' if torch.cuda.is_available() else 'cpu'}")

[OK] Imports completados y path configurado.
Dispositivo actual: cuda


## 2. Configuración del Pipeline y Construcción de Secuencias

Cargamos los datos fisiológicos crudos (ECG, EDA, Resp, EEG) desde el dataset `fatigueset`, alineamos los streams temporales mediante `merge_asof` e interpolamos valores perdidos para construir tensores de secuencias temporales coherentes.

In [2]:
# Configuración del dataset y pipeline
dataset_path = str(Path.cwd().parent / "fatigueset")
pipeline = FatigueSetPipeline(dataset_path=dataset_path, umbral_nulos=5.0)

print("Cargando dataset...")
raw = pipeline.cargar_dataset(verbose=False)

print("Preparando targets del dataframe ML...")
df_ml = pipeline.construir_dataset_ml(raw)
df_targets = _prepare_target_table(df_ml)

print("Combinando streams fisiológicos crudos (Chest y Wrist)...")
df_raw = _merge_raw_streams(raw)

# Parámetros de ventanas de secuencia temporal
seq_len = 128
step = 32

print(f"Construyendo secuencias de tamaño={seq_len} y paso={step}...")
X_arr, y_arr, groups, feature_cols = _build_sequences(
    df_raw=df_raw,
    df_targets=df_targets,
    seq_len=seq_len,
    step=step
)

print(f"[OK] Dimensiones de tensores construidos:")
print(f"  - X: {X_arr.shape} (Número de ventanas x seq_len x features)")
print(f"  - y: {y_arr.shape} (Número de ventanas x 2 targets)")
print(f"  - Columnas de sensores: {len(feature_cols)}")

Cargando dataset...


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\fatigueset-lib\fatigueset\loader.py:255: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(dfs, ignore_index=True) if dfs else None


Preparando targets del dataframe ML...


Combinando streams fisiológicos crudos (Chest y Wrist)...


Construyendo secuencias de tamaño=128 y paso=32...
[OK] Dimensiones de tensores construidos:
  - X: (1306, 128, 23) (Número de ventanas x seq_len x features)
  - y: (1306, 2) (Número de ventanas x 2 targets)
  - Columnas de sensores: 23


## 3. División de Datos por Participante (Group Split) y DataLoaders

Para garantizar la rigurosidad científica y evitar la fuga de información (data leakage), separamos el dataset de tal manera que el sujeto utilizado para validar/probar nunca esté presente en el conjunto de entrenamiento.

In [3]:
# Hacemos una partición de prueba donde el participante '01' se reserva para validación
train_idx = np.where(groups != '01')[0]
val_idx = np.where(groups == '01')[0]

X_train, y_train = X_arr[train_idx], y_arr[train_idx]
X_val, y_val = X_arr[val_idx], y_arr[val_idx]

train_dataset = FatigueSequenceDataset(X_train, y_train)
val_dataset = FatigueSequenceDataset(X_val, y_val)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

Train samples: 1184
Validation samples: 122


## 4. Inicialización del Regresor GRU Manual

Instanciamos nuestro modelo regresor `CustomGRURegressor` usando el número de características predictivas fisiológicas detectadas. El modelo predice simultáneamente los niveles continuos de `fatiga_fisica` y `fatiga_mental` (salida bidimensional).

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = len(feature_cols)
hidden_size = 64
num_layers = 2
dropout = 0.2

model = CustomGRURegressor(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    dropout=dropout,
    output_size=2
).to(device)

print(model)

CustomGRURegressor(
  (gru): CustomGRU(
    (layers): ModuleList(
      (0-1): 2 x CustomGRUCell()
    )
    (dropout_layer): Dropout(p=0.2, inplace=False)
  )
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


## 5. Entrenamiento Corto de Validación (Sanity Check)

Ejecutamos un entrenamiento corto de 5 épocas sobre el conjunto de datos para verificar la estabilidad de las operaciones de tensores en las celdas GRU y del motor común de optimización.

In [5]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
print("Iniciando mini-entrenamiento...")

for epoch in range(1, epochs + 1):
    # Modo entrenamiento
    model.train()
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        
        # Gradient clipping para evitar inestabilidad recurrente
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_train_loss += loss.item()
        
    avg_train_loss = total_train_loss / len(train_loader)
    
    # Modo validación
    model.eval()
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
    
    print(f"Epoch {epoch}/{epochs} - Train Loss (MSE): {avg_train_loss:.6f} - Val Loss (MSE): {avg_val_loss:.6f}")

print("[OK] Entrenamiento corto finalizado correctamente.")

Iniciando mini-entrenamiento...


Epoch 1/5 - Train Loss (MSE): 1200.165346 - Val Loss (MSE): 1097.226517


Epoch 2/5 - Train Loss (MSE): 1023.749820 - Val Loss (MSE): 933.432983


Epoch 3/5 - Train Loss (MSE): 901.961508 - Val Loss (MSE): 792.341736


Epoch 4/5 - Train Loss (MSE): 795.442799 - Val Loss (MSE): 667.316711


Epoch 5/5 - Train Loss (MSE): 704.967071 - Val Loss (MSE): 556.175568
[OK] Entrenamiento corto finalizado correctamente.


## 6. Serialización del Modelo de Fatiga

Persistimos el estado del modelo en la carpeta centralizada de modelos del proyecto `/models/` bajo la categoría correspondiente.

In [6]:
output_dir = Path.cwd().parent / "models" / "deep_learning"
output_dir.mkdir(parents=True, exist_ok=True)

model_path = output_dir / "gru_fatigue_notebook.pt"
torch.save(model.state_dict(), model_path)

print(f"[OK] Modelo guardado exitosamente en: {model_path}")

[OK] Modelo guardado exitosamente en: C:\Users\egull\OneDrive\Documentos\Proyectos\tfg\models\deep_learning\gru_fatigue_notebook.pt
